In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


## Env

In [2]:
from src.utils.env import prepare_environment

prepare_environment("../api_keys")

INFO 08-31 17:11:28 [src.utils.env:30] Setting API tokens: ['WANDB_API_KEY', 'OPENPIPE_API_KEY', 'OPENAI_API_KEY']
INFO 08-31 17:11:28 [src.utils.env:44] Setting additional variables: {'NCCL_CUMEM_ENABLE': '0', 'VLLM_USE_V1': '0', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn', 'ART_SERVER_TIMEOUT': '300', 'WEAVE_DISABLED': '1', 'WEAVE_DISABLE_TRACING': '1'}


## Model

In [3]:
import src.configs.models.art as art_configs
import src.configs.models.vllm as vllm_configs
from src.configs import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

base_model = "unsloth/Qwen3-32B"
project_name = "test"

path_config = PathConfig(
    base_model=base_model,
    project_name=project_name,
)

Available ART model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B', 'unsloth/Qwen3-32B']
Available vLLM model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B', 'unsloth/Qwen3-32B', 'unsloth/Qwen3-32B-unsloth-bnb-4bit']


In [4]:
import art

host = "0.0.0.0"
port = 8001

model = art.Model(
    name=base_model,
    project=path_config.project_name,
    inference_api_key=os.getenv("OPENAI_API_KEY", "default"),
    inference_base_url=f"https://{host}:{port}/v1",
)

## Data

In [5]:
# from datasets import Dataset, load_dataset, DatasetDict

# dataset_dict: DatasetDict = load_dataset(
#     path="furonghuang-lab/Easy2Hard-Bench",
#     name="E2H-AMC",
#     split=None,
# )  # type: ignore

# train_data: Dataset = dataset_dict["train"]
# test_data: Dataset = dataset_dict["eval"]

In [6]:
from datasets import Dataset, load_dataset, DatasetDict


def load_data() -> tuple[Dataset, Dataset]:
    data: Dataset = load_dataset(
        path="BBEH/bbeh",
        split="train",
    )  # type: ignore

    # Filter by tasks
    data = data.map(lambda sample: {"task": sample["task"].replace(" ", "_")})
    data = data.filter(lambda sample: sample["task"] == "boolean_expressions")

    split_dict = data.train_test_split(test_size=0.25, seed=0)
    ds_train = split_dict["train"]
    ds_eval = split_dict["test"]
    return ds_train, ds_eval


ds_train, ds_eval = load_data()

## Inference Clients

In [7]:
from src.vllm_client import VllmClient, VllmRouter

inference_clients: list[VllmClient] = []
vllm_server_ports = [port]
for port in vllm_server_ports:
    inference_clients.append(VllmClient.from_connection(port=port, base_model=base_model))

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Config 

In [19]:
from experiments.bbeh.trainer import BbehTrainer
from src.configs import PromptConfig, DecompConfig, RolloutConfig

prompt_config = PromptConfig(
    system_root="",
    system_inter="",
    system_leaf="",
    tasks_depleted="",
)

# prompt_config = PromptConfig(
#     system_root="",
#     system_inter="",
#     system_leaf="",
# )

decomp_config = DecompConfig(
    max_depth=1,
    max_tasks=4,
    max_rounds=4,
)

rollout_config = RolloutConfig(
    kwargs={
        "max_completion_tokens": 4500,
        "temperature": 0.7,
        "top_p": 0.8,
    }
)

if "Qwen3" in base_model:
    rollout_config.kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False}}

trainer = BbehTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    decomp_config=decomp_config,
    rollout_config=rollout_config,
)

In [20]:
from src.trainer import RolloutStage

result = await trainer.rollout(
    dataset=ds_eval.to_list(),
    group_size=1,
    stage=RolloutStage.Val,
)

rollout-val:   0%|          | 0/50 [00:00<?, ?it/s]

In [10]:
result_trajectories = [g.trajectories[0] for g in result ]

In [14]:
from src.utils.analysis import to_dataframe, average_metrics

df = to_dataframe(result_trajectories)

In [15]:
print(average_metrics(result_trajectories))

{'direct_calls': 1.0, 'total_calls': 1.0, 'direct_tasks': 0.0, 'total_tasks': 0.0, 'max_depth': 0.0, 'total_tokens': 2578.28, 'response_completed': 0.0, 'duration': 47.51309019999999, 'answer_reward': 0.12, 'format_reward': -1.0, 'behavior_reward': -1.0, 'is_correct': 0.04, 'gave_answer': 0.0, 'parse_success': 0.22, 'completion_tokens': 1000.0}
